# Project Oracle Walkthrough

This notebook demonstrates the core microstructure analytics services
that power Confluence Decoder's real-time options analysis.

In [ ]:
import sys
sys.path.insert(0, 'backend')

import numpy as np
import matplotlib.pyplot as plt
from services.gex_aggregator import GexAggregator
from services.vpin_engine import VpinEngine
from services.stochastic_vol import SABRModel, VolSurfaceConstructor
from services.trinity_alignment import TrinityAlignmentIndex
from services.anomaly_detector import StatisticalAnomalyDetector
from services.liquidity_metrics import MarketFragilityIndex

plt.style.use('dark_background')
np.random.seed(42)
print('All services imported successfully.')

## Cell 1: Compute GEX from a Synthetic Chain -> Heatmap

Build a synthetic options chain, compute the GEX surface, and visualize
it as a strike x expiry heatmap.

In [ ]:
spot = 500.0
strikes = np.arange(450, 551, 5)
expiries = [0.1, 0.25, 0.5, 1.0]

contracts = []
for K in strikes:
    for T in expiries:
        gamma = 0.02 * np.exp(-0.5 * ((K - spot) / 20) ** 2)
        oi = 100 + int(500 * np.exp(-0.5 * ((K - spot) / 15) ** 2))
        ctype = 'call' if K <= spot else 'put'
        contracts.append({
            'strike': float(K), 'gamma': float(gamma),
            'oi': float(oi), 'type': ctype, 'expiry': T
        })

agg = GexAggregator()
result = agg.compute(spot, contracts)

gex_surface = np.array(result['gex_surface'])
fig, ax = plt.subplots(figsize=(10, 6))
im = ax.imshow(gex_surface.T, aspect='auto', cmap='RdYlGn',
               origin='lower', interpolation='nearest')
ax.set_yticks(range(len(result['expiries'])))
ax.set_yticklabels(['T={:.2f}y'.format(T) for T in result['expiries']])
ax.set_xticks(range(0, len(result['strikes']), 5))
ax.set_xticklabels(['{:.0f}'.format(s) for s in result['strikes'][::5]])
ax.set_xlabel('Strike')
ax.set_ylabel('Expiry')
ax.set_title('GEX Surface (Strike x Expiry)')
plt.colorbar(im, label='Signed GEX ($)')
plt.tight_layout()
plt.show()

print('Net GEX: ${:,.0f}'.format(result['net_gex']))
print('Zero-gamma levels:', result['zero_gamma_levels'])

## Cell 2: VPIN Engine with Synthetic Ticks -> Toxicity Time Series

Simulate a stream of trades with alternating buy/sell pressure and
track the VPIN (Volume-Synchronized PIN) over time.

In [ ]:
eng = VpinEngine(bucket_size=500.0, window=20)
rng = np.random.default_rng(42)

vpin_history = []
n_trades = 300

for i in range(n_trades):
    if i < 150:
        dp = rng.normal(0, 0.3)
    else:
        dp = rng.normal(0.8, 0.2)
    vol = rng.uniform(100, 500)
    sigma = 0.15
    eng.update(price_change=dp, volume=vol, sigma=sigma, dt=1.0)
    state = eng.get_state()
    if state['current']['vpin'] is not None:
        vpin_history.append(state['current']['vpin'])

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(vpin_history, color='cyan', linewidth=1.5)
ax.axvline(x=150, color='red', linestyle='--', alpha=0.7, label='Toxicity starts')
ax.axhline(y=0.5, color='yellow', linestyle=':', alpha=0.5, label='VPIN=0.5')
ax.set_xlabel('Trade #')
ax.set_ylabel('VPIN')
ax.set_title('VPIN Toxicity Time Series')
ax.legend()
ax.set_ylim(0, 1)
plt.tight_layout()
plt.show()

print('Mean VPIN (first 150): {:.3f}'.format(np.mean(vpin_history[:150])))
print('Mean VPIN (last 150): {:.3f}'.format(np.mean(vpin_history[150:])))

## Cell 3: SABR Fit to a Real Vol Smile

Generate a synthetic vol smile using known SABR parameters, then
fit the model and compare fitted vs observed vols.

In [ ]:
F = 500.0
T = 0.25
true_alpha, true_beta, true_rho, true_nu = 0.22, 0.7, -0.4, 0.5

true_model = SABRModel(alpha=true_alpha, beta=true_beta,
                        rho=true_rho, nu=true_nu)

strikes = np.linspace(450, 550, 21)
true_vols = [true_model.hagan_lognormal_vol(F, K, T) for K in strikes]

rng = np.random.default_rng(123)
market_vols = [v + rng.normal(0, 0.005) for v in true_vols]

fit_model = SABRModel(alpha=0.2, beta=0.5, rho=0.0, nu=0.3)
fit_result = fit_model.fit(strikes, market_vols, F, T)
fitted_vols = [fit_model.hagan_lognormal_vol(F, K, T) for K in strikes]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(strikes, true_vols, 'o-', label='True', color='cyan')
ax1.plot(strikes, market_vols, 'x', label='Market (noisy)', color='yellow', alpha=0.7)
ax1.plot(strikes, fitted_vols, '--', label='Fitted', color='magenta')
ax1.set_xlabel('Strike')
ax1.set_ylabel('Implied Vol')
ax1.set_title('SABR Vol Smile Fit')
ax1.legend()
ax1.axvline(F, color='white', linestyle=':', alpha=0.3)

residuals = np.array(fitted_vols) - np.array(market_vols)
ax2.bar(strikes, residuals, width=3, color='green', alpha=0.7)
ax2.axhline(0, color='white', linestyle='-')
ax2.set_xlabel('Strike')
ax2.set_ylabel('Residual (Fitted - Market)')
ax2.set_title('Fit Residuals')

plt.tight_layout()
plt.show()

print('True params:  alpha={:.3f}, beta={:.3f}, rho={:.3f}, nu={:.3f}'.format(
    true_alpha, true_beta, true_rho, true_nu))
print('Fitted params: alpha={:.3f}, beta={:.3f}, rho={:.3f}, nu={:.3f}'.format(
    fit_result['alpha'], fit_result['beta'], fit_result['rho'], fit_result['nu']))
print('RMSE: {:.6f}'.format(fit_result['rmse']))

## Cell 4: Trinity Alignment with Synthetic ZG Levels

Compute the Trinity Alignment Index for SPX/SPY/QQQ zero-gamma levels
and interpret the confluence score.

In [ ]:
ta = TrinityAlignmentIndex(tolerance_pct=0.005)

result_strong = ta.compute(
    spy_flip_levels=[500.0, 505.0],
    qqq_flip_levels=[500.5],
    spx_flip_levels=[5000.0, 5050.0],
    spy_spot=502.0, qqq_spot=503.0, spx_spot=5020.0
)

result_weak = ta.compute(
    spy_flip_levels=[400.0],
    qqq_flip_levels=[600.0],
    spx_flip_levels=[8000.0],
    spy_spot=500.0, qqq_spot=500.0, spx_spot=5000.0
)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

scenarios = ['Strong\nAlignment', 'Weak\nAlignment']
scores = [result_strong['score'], result_weak['score']]
regimes = [result_strong['regime'], result_weak['regime']]
colors = ['green' if s >= 50 else 'red' for s in scores]

bars = ax1.bar(scenarios, scores, color=colors, alpha=0.8)
ax1.set_ylim(0, 100)
ax1.set_ylabel('Trinity Score')
ax1.set_title('Trinity Alignment Index')
ax1.axhline(y=75, color='green', linestyle=':', alpha=0.5, label='STRONG threshold')
ax1.axhline(y=25, color='red', linestyle=':', alpha=0.5, label='NONE threshold')
ax1.legend(fontsize=8)

for bar, score, regime in zip(bars, scores, regimes):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 2,
             '{:.1f}\n({})'.format(score, regime), ha='center', va='bottom', fontsize=10)

ax2.axis('off')
text = 'Strong Alignment Details:\n'
text += '  Score: {:.1f} ({})\n'.format(result_strong['score'], result_strong['regime'])
text += '  Aligned levels: {}\n'.format(len(result_strong['aligned_levels']))
for al in result_strong['aligned_levels']:
    text += '    Level {:.1f}: {} (spread={:.2f}%)\n'.format(
        al['level'], al['instruments'], al['spread_pct'])
ax2.text(0.1, 0.5, text, transform=ax2.transAxes, fontsize=10,
         verticalalignment='center', fontfamily='monospace')

plt.tight_layout()
plt.show()

## Cell 5: Anomaly Detector -> Injected Toxicity -> Score Crossing Threshold

Feed normal data to warm up the detector, then inject anomalous toxicity
spikes and observe the anomaly score crossing the adaptive threshold.

In [ ]:
det = StatisticalAnomalyDetector(window=100, threshold_sigma=2.0)
rng = np.random.default_rng(42)

scores = []
thresholds = []
is_anomaly = []
n_normal = 80
n_anomaly = 20

for i in range(n_normal + n_anomaly):
    if i < n_normal:
        features = rng.normal(0.5, 0.05, 2)
    else:
        features = np.array([rng.uniform(5.0, 10.0), rng.uniform(-10.0, -5.0)])
    result = det.update(features)
    scores.append(result['anomaly_score'])
    thresholds.append(result['threshold'])
    is_anomaly.append(result['is_anomaly'])

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(scores, label='Anomaly Score', color='cyan', linewidth=1.5)
ax.plot(thresholds, label='Threshold', color='red', linewidth=1, linestyle='--')
ax.axvline(x=n_normal, color='yellow', linestyle=':', alpha=0.7, label='Anomaly injection starts')

for i in range(len(scores)):
    if is_anomaly[i]:
        ax.axvspan(i-0.5, i+0.5, alpha=0.2, color='red')

ax.set_xlabel('Observation #')
ax.set_ylabel('Score')
ax.set_title('Anomaly Detector: Injected Toxicity Detection')
ax.legend()
plt.tight_layout()
plt.show()

n_detected = sum(is_anomaly[n_normal:])
print('Anomalies injected: {}'.format(n_anomaly))
print('Anomalies detected: {}'.format(n_detected))
print('Recall: {:.0%}'.format(n_detected/n_anomaly))